In [16]:
from google.colab import drive
import os

In [17]:
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/kalanidhi.zip" "/content/kalanidhi.zip"
!unzip -o /content/kalanidhi.zip -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archive:  /content/kalanidhi.zip
  inflating: /content/data/tokenizer/pruned_config.json  
  inflating: /content/src/data/loader.py  
  inflating: /content/src/data/__pycache__/loader.cpython-314.pyc  
  inflating: /content/src/diffusion/engine.py  
  inflating: /content/src/model/config.py  
  inflating: /content/src/model/transformer.py  
  inflating: /content/src/model/__pycache__/config.cpython-314.pyc  
  inflating: /content/src/model/__pycache__/transformer.cpython-314.pyc  
  inflating: /content/src/tokenizer/prune.py  
  inflating: /content/kalanidhi_launcher.ipynb  


In [18]:
import os

# 1. Create the folder
os.makedirs('/content/training', exist_ok=True)

# 2. Create the __init__.py so Python treats it as a package
with open('/content/training/__init__.py', 'w') as f:
    pass

print(" 'training' folder created.")

 'training' folder created.


In [25]:
%%writefile /content/src/model/transformer.py
import torch
import torch.nn as nn
from src.model.config import KalanidhiConfig

class KalanidhiModel(nn.Module):
    def __init__(self, config: KalanidhiConfig):
        super().__init__()
        self.config = config

        self.token_emb = nn.Embedding(config.vocab_size, config.hidden_size)
        self.pos_emb = nn.Embedding(config.max_position_embeddings, config.hidden_size)
        self.embed_dropout = nn.Dropout(config.hidden_dropout_prob)

        self.time_mlp = nn.Sequential(
            nn.Linear(1, config.hidden_size),
            nn.SiLU(),
            nn.Linear(config.hidden_size, config.hidden_size)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.hidden_size,
            nhead=config.num_attention_heads,
            dim_feedforward=config.intermediate_size,
            dropout=config.attention_probs_dropout_prob,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=config.num_hidden_layers)

        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

    def forward(self, input_ids, t, attention_mask=None):
        if t.dim() == 1:
            t = t.unsqueeze(-1)

        t_emb = self.time_mlp(t).unsqueeze(1)

        positions = torch.arange(input_ids.size(1), device=input_ids.device)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        x = self.embed_dropout(x)

        src_key_padding_mask = None
        if attention_mask is not None:
            src_key_padding_mask = (attention_mask == 0)

        x = self.transformer(x + t_emb, src_key_padding_mask=src_key_padding_mask)

        return self.lm_head(x)

Overwriting /content/src/model/transformer.py


In [26]:
import os

cache = "/content/src/model/__pycache__/transformer.cpython-314.pyc"
if os.path.exists(cache):
    os.remove(cache)
    print("Cache cleared.")

Cache cleared.


In [27]:
import importlib
import src.model.transformer as t_module
importlib.reload(t_module)
from src.model.transformer import KalanidhiModel
print("KalanidhiModel reloaded.")

KalanidhiModel reloaded.


In [ ]:
import os
from huggingface_hub import login

# Get token from environment variable
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    login(hf_token)
else:
    print("Warning: HF_TOKEN not set. Add it as a Colab secret for access to gated models.")

In [35]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from src.model.config import KalanidhiConfig
from src.model.transformer import KalanidhiModel
from src.diffusion.engine import KalanidhiDiffusion
from src.data.loader import KalanidhiDataset

def resume_train(start_epoch=10, total_epochs=40, batch_size=32, lr=1e-4, max_length=256):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Resuming training on {device} from epoch {start_epoch}...")

    config = KalanidhiConfig()
    model = KalanidhiModel(config).to(device)
    diffusion = KalanidhiDiffusion()

    checkpoint = torch.load(f"/content/drive/MyDrive/kalanidhi_epoch_{start_epoch}.pt", map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    dataset = KalanidhiDataset(max_length=max_length)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    loss_fn = nn.CrossEntropyLoss(ignore_index=0)

    model.train()
    print(f"Dataset size: {len(dataset)} samples, {len(loader)} batches/epoch.")

    for epoch in range(start_epoch, total_epochs):
        total_loss = 0.0

        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            t = diffusion.sample_t(input_ids.size(0), device)
            noised_ids, noise_mask = diffusion.apply_noise(input_ids, t)

            logits = model(noised_ids, t, attention_mask=attention_mask)

            logits_flat = logits[noise_mask]
            targets_flat = input_ids[noise_mask]

            loss = loss_fn(logits_flat, targets_flat)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        print(f"Epoch {epoch + 1}/{total_epochs} - Loss: {avg_loss:.4f}")

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": avg_loss,
        }, f"/content/drive/MyDrive/kalanidhi_epoch_{epoch + 1}.pt")
        print(f"Checkpoint saved for epoch {epoch + 1}.")

    print("Training complete.")

resume_train()

Resuming training on cuda from epoch 10...
Loading Telugu TinyStories...
Dataset size: 10924 samples, 342 batches/epoch.
Epoch 11/40 - Loss: 4.9706
Checkpoint saved for epoch 11.
Epoch 12/40 - Loss: 4.9547
Checkpoint saved for epoch 12.
Epoch 13/40 - Loss: 4.9342
Checkpoint saved for epoch 13.
Epoch 14/40 - Loss: 4.9218
Checkpoint saved for epoch 14.
Epoch 15/40 - Loss: 4.9154
Checkpoint saved for epoch 15.
Epoch 16/40 - Loss: 4.9025
Checkpoint saved for epoch 16.
Epoch 17/40 - Loss: 4.8886
Checkpoint saved for epoch 17.
Epoch 18/40 - Loss: 4.8804
Checkpoint saved for epoch 18.
Epoch 19/40 - Loss: 4.8682
Checkpoint saved for epoch 19.
Epoch 20/40 - Loss: 4.8580
Checkpoint saved for epoch 20.
Epoch 21/40 - Loss: 4.8494
Checkpoint saved for epoch 21.
Epoch 22/40 - Loss: 4.8367
Checkpoint saved for epoch 22.
Epoch 23/40 - Loss: 4.8303
Checkpoint saved for epoch 23.
Epoch 24/40 - Loss: 4.8219
Checkpoint saved for epoch 24.
Epoch 25/40 - Loss: 4.8183
Checkpoint saved for epoch 25.
Epoch 26/

In [37]:
import os
for i in range(1, 41):
    path = f"/content/drive/MyDrive/kalanidhi_epoch_{i}.pt"
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024
        print(f"Epoch {i}: {size:.2f} MB")
    else:
        print(f"Epoch {i}: missing")

Epoch 1: 58.07 MB
Epoch 2: 58.07 MB
Epoch 3: 58.07 MB
Epoch 4: 58.07 MB
Epoch 5: 58.07 MB
Epoch 6: 58.07 MB
Epoch 7: 58.07 MB
Epoch 8: 58.07 MB
Epoch 9: 58.07 MB
Epoch 10: 58.07 MB
Epoch 11: 58.07 MB
Epoch 12: 58.07 MB
Epoch 13: 58.07 MB
Epoch 14: 58.07 MB
Epoch 15: 58.07 MB
Epoch 16: 58.07 MB
Epoch 17: 58.07 MB
Epoch 18: 58.07 MB
Epoch 19: 58.07 MB
Epoch 20: 58.07 MB
Epoch 21: 58.07 MB
Epoch 22: 58.07 MB
Epoch 23: 58.07 MB
Epoch 24: 58.07 MB
Epoch 25: 58.07 MB
Epoch 26: 58.07 MB
Epoch 27: 58.07 MB
Epoch 28: 58.07 MB
Epoch 29: 58.07 MB
Epoch 30: 58.07 MB
Epoch 31: 58.07 MB
Epoch 32: 58.07 MB
Epoch 33: 58.07 MB
Epoch 34: 58.07 MB
Epoch 35: 58.07 MB
Epoch 36: 58.07 MB
Epoch 37: 58.07 MB
Epoch 38: 58.07 MB
Epoch 39: 58.07 MB
Epoch 40: 58.07 MB


In [38]:
import json
import torch
from pathlib import Path
from transformers import AutoTokenizer
from src.model.transformer import KalanidhiModel
from src.model.config import KalanidhiConfig
from src.diffusion.engine import KalanidhiDiffusion

def generate_sample(num_steps=20, max_length=32, temperature=0.3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    config_path = Path("/content/data/tokenizer/pruned_config.json")
    with open(config_path) as f:
        pruned = json.load(f)

    old_to_new = {int(k): v for k, v in pruned["old_to_new"].items()}
    new_to_old = {v: k for k, v in old_to_new.items()}

    tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")

    config = KalanidhiConfig()
    model = KalanidhiModel(config).to(device)
    checkpoint = torch.load("/content/drive/MyDrive/kalanidhi_epoch_10.pt", map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    diffusion = KalanidhiDiffusion()

    cls_id = old_to_new.get(tokenizer.cls_token_id, 2)
    sep_id = old_to_new.get(tokenizer.sep_token_id, 3)
    mask_id = diffusion.mask_token_id

    tokens = [mask_id] * max_length
    tokens[0] = cls_id
    tokens[-1] = sep_id
    input_ids = torch.tensor([tokens], dtype=torch.long).to(device)

    print("Denoising Telugu sequence...")
    with torch.no_grad():
        for step in range(num_steps):
            t_val = 1.0 - (step / num_steps)
            t = torch.tensor([t_val], device=device)

            logits = model(input_ids, t)
            probs = torch.softmax(logits / temperature, dim=-1)
            predicted_ids = torch.multinomial(probs.view(-1, config.vocab_size), 1).view(1, -1)

            mask_positions = (input_ids == mask_id)
            input_ids[mask_positions] = predicted_ids[mask_positions]

    pruned_ids = input_ids[0].tolist()
    original_ids = [int(new_to_old.get(tid, tokenizer.unk_token_id)) for tid in pruned_ids]
    text = tokenizer.decode(original_ids, skip_special_tokens=True)
    print(f"Kalanidhi says: {text}")

generate_sample()

Denoising Telugu sequence...
Kalanidhi says: త : నసనత.త" న.త.షచ,తనత. చదసతన.స
